# 08.4 EnCodec：把波形变成离散 token

本 Notebook 展示 neural audio codec 的核心接口：输入波形，得到多码本离散 token，再从 token 重建音频。当前环境若未安装 `encodec`，Notebook 会打印安装说明并跳过主流程。安装后不需要改 Notebook 即可运行 encode/decode。


In [ ]:
from pathlib import Path
import json
import os
import sys

# 路径推断：从 cwd 向上找含 CODE/chapter08/_common 的目录；ROOT 指向 CODE/chapter08/
_p = Path.cwd()
while not (_p / "CODE" / "chapter08" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter08/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter08"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Audio, display

from _common.audio_io import load_audio
from _common.dataset_registry import check_required_assets, load_nsynth_metadata
from _common.device_utils import choose_device
from _common.paths import portable_path
from _common.plotting import plot_spectrogram, setup_plot_style
from _common.tables import write_rows
from codec.encodec_adapter import EncodecAdapter, check_encodec, encodec_output_paths
from codec.reconstruction import reconstruction_mse, reconstruction_snr_db
from codec.token_visualization import plot_codebook_usage, plot_token_matrix

OUTPUT_FIGURES = ROOT / "output_figures"
OUTPUT_AUDIO = ROOT / "output_audio" / "08_4"
OUTPUT_TOKENS = ROOT / "outputs" / "generated" / "encodec_tokens"
OUTPUT_TABLES = ROOT / "outputs" / "tables"
for path in [OUTPUT_FIGURES, OUTPUT_AUDIO, OUTPUT_TOKENS, OUTPUT_TABLES]:
    path.mkdir(parents=True, exist_ok=True)
setup_plot_style()
DEVICE = choose_device(os.getenv("CHAPTER08_DEVICE", "auto"))
print("selected device:", DEVICE)

def rel(path):
    return portable_path(path, ROOT)


In [ ]:
check_required_assets(
    ["nsynth"],
    message="08_4 uses a short NSynth WAV as the EnCodec input example.",
    stop=True,
)
rows = load_nsynth_metadata("valid", limit=1)
sample_audio_path = Path(rows[0]["audio_path"])
print("sample:", rel(sample_audio_path))
display(Audio(str(sample_audio_path)))


In [ ]:
status = check_encodec()
print(status)
ENCODEC_AVAILABLE = status.available
if not ENCODEC_AVAILABLE:
    print("Install command:", status.next_action)


In [ ]:
if ENCODEC_AVAILABLE:
    import torch

    adapter = EncodecAdapter(model_name="24khz", bandwidth=6.0, device=DEVICE)
    result = adapter.encode_decode_file(sample_audio_path)
    paths = encodec_output_paths(sample_audio_path, OUTPUT_TOKENS)
    torch.save(result.codes, paths["tokens"])
    paths["metadata"].write_text(
        json.dumps(
            {
                "source_audio": rel(sample_audio_path),
                "sample_rate": result.sample_rate,
                "bandwidth": result.bandwidth,
                "codes_shape": list(result.codes.shape),
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    from _common.audio_io import save_audio

    reconstruction_path = OUTPUT_AUDIO / f"{sample_audio_path.stem}.encodec_reconstruction.wav"
    save_audio(reconstruction_path, result.reconstruction, result.sample_rate)
    print("codes shape:", tuple(result.codes.shape))
    display(Audio(str(reconstruction_path)))
else:
    result = None
    reconstruction_path = None


In [ ]:
if ENCODEC_AVAILABLE:
    codes = result.codes.squeeze(0).numpy()
    fig = plot_token_matrix(codes, OUTPUT_FIGURES / "08_4_encodec_token_matrix.png")
    plt.show()
    fig = plot_codebook_usage(codes, OUTPUT_FIGURES / "08_4_encodec_codebook_usage.png")
    plt.show()
else:
    print("Skip token plots because EnCodec is not installed.")


In [ ]:
if ENCODEC_AVAILABLE:
    reference, reference_sr = load_audio(sample_audio_path, sr=result.sample_rate, mono=True)
    metrics = [
        {
            "source_audio": rel(sample_audio_path),
            "reconstruction": rel(reconstruction_path),
            "sample_rate": result.sample_rate,
            "bandwidth": result.bandwidth,
            "mse": reconstruction_mse(reference, result.reconstruction),
            "snr_db": reconstruction_snr_db(reference, result.reconstruction),
            "codes_shape": "x".join(str(x) for x in result.codes.shape),
        }
    ]
    write_rows(OUTPUT_TABLES / "08_4_encodec_reconstruction_metrics.csv", metrics)
    display(pd.DataFrame(metrics))

    fig = plot_spectrogram(
        result.reconstruction,
        result.sample_rate,
        title="EnCodec 重建音频",
        out_path=OUTPUT_FIGURES / "08_4_encodec_reconstruction_spectrogram.png",
    )
    plt.show()
else:
    print("Skip reconstruction metrics because EnCodec is not installed.")


In [ ]:
print("08_4 status:", "ran encode/decode" if ENCODEC_AVAILABLE else "waiting for encodec dependency")
for path in sorted(OUTPUT_FIGURES.glob("08_4_*.png")):
    print("-", rel(path))
